# E5-small: дообучение поиска приложений

Сравниваем готовую E5 с одной адаптацией на нашем датасете. Encoder отдельно читает `query:` и `passage:`, строит mean-pooled L2 embeddings. Новой головы нет; меняются общие веса encoder. Карточки после обучения нужно индексировать заново.

## Протокол до просмотра результатов

22 656 прежних train (5 664 clean + 16 992 STT); первые 2 272 чистые, затем смесь. Максимум один проход; all weights FP32, lr=1e-5, decay=.01, effective16/micro2, warmup6%, clip1. Train сравнивает цель с 19 закреплёнными соседями исходной E5: per-query CE по cosine/0.01. Чужие train-запросы не создают дополнительные ложные negatives.

Выбор весов: прежние150 synthetic validation, но **весь177-объектный каталог**. Группы/профили имеют равный вес; NLL каждые142 обновления, patience3/min_delta.005. Исходные веса тоже eligible. Никакой real, transfer или independent test на Kaggle нет.

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.12.1', 'safetensors==0.6.2'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU Kaggle'
print({'torch':torch.__version__, 'gpu':torch.cuda.get_device_name(0)})


## Что проверяет validation

E5 ранжирует весь каталог; accuracy, recall@20 и MRR считаются по всем177 кандидатам. Softmax с temperature=.01 нужен для NLL, а не как проверенная вероятность правильного открытия. Тексты длиннее512 токенов прерывают запуск; metadata, ID и target не добавляются к запросу.

In [ ]:
matches = list(Path('/kaggle/input').rglob('e5-input-manifest.json'))
assert len(matches)==1, f'Ожидался один пакет, найдено {len(matches)}'
input_dir = matches[0].parent
assert hashlib.sha256(matches[0].read_bytes()).hexdigest() == '709b923a01045872dc685d9d0df09ed9cb8bdd4ec49760f80293a99a6e3828a8'
manifest = json.loads(matches[0].read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((input_dir/name).read_bytes()).hexdigest()==expected, name
sys.path.insert(0, str(input_dir))
from types import SimpleNamespace
from train_e5_kaggle import run
result = run(SimpleNamespace(input=input_dir, output=Path('/kaggle/working/results'), seed=20261005, smoke=False))


## Результат и локальная проверка

Сохраняем выбранные/последние веса и ledgers отдельно. После выбора — известные800 представлений с полными каталогами, парные ошибки исходной/новой E5 и CPU-скорость с пересчитанным индексом. В приложении Laya остаётся основной. Известный test не становится новым слепым подтверждением.

In [ ]:
print({k:result[k] for k in ('steps','best_step','stop_reason','elapsed_seconds')})
for profile, values in result['best']['metrics'].items():
    print(profile, values)
